# Dia 07 — CSV

**🃏 Cartas:** 7 · 28 · 49 &nbsp;·&nbsp; **📘 Original:** *Day 7. CSV*
&nbsp;·&nbsp; [repositório](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias) &nbsp;·&nbsp; [curso original do cipher387](https://github.com/cipher387/python-for-OSINT-21-days)

---

## 📖 Capítulo 7 — A planilha das vítimas

Quarta-feira, 10h. A ficha em JSON de ontem já tem duas mentiras registradas: o endereço copiado e o
CNPJ inventado. A Iara abre na tela grande a planilha que a delegacia mandou: **37 denúncias**, uma
por linha, separadas por ponto e vírgula.

— Isso aqui foi digitado por quatro escrivães diferentes, em três turnos. Olha a coluna de cidade.

Você olha. "Belo Horizonte", "belo horizonte", "Belo Horizone". "Contagem" e "contagem".
"Poços de Caldas" e "Pocos de Caldas". Para uma pessoa, é tudo a mesma coisa. Para o computador, são
cidades diferentes.

— Se você contar vítimas por cidade agora, BH vai aparecer dividida em três, e o relatório vai dizer
que o golpe foi menor do que foi. **Dado sujo mente.** E tem mais: some o prejuízo com cuidado.
Centavo errado em relatório de fraude é pergunta do advogado de defesa.

Ela abre outra aba: a lista oficial de municípios do IBGE.

— A gente não inventa nome de cidade. A gente compara com a fonte oficial, corrige o que dá para
corrigir, e marca o resto para um humano olhar. E nunca, **nunca** apaga a coluna original.

In [ ]:
#@title ⚙️ Preparar o ambiente (rode esta célula primeiro)
# No Colab: baixa o repositório e entra na pasta do dia.
# No Jupyter local: o notebook já está na pasta do dia, então não faz nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias"
if "google.colab" in sys.modules:
    destino = "/content/python-para-osint-21-dias"
    if not os.path.isdir(destino):
        subprocess.run(["git", "clone", "--depth", "1", REPO, destino], check=True)
    os.chdir(f"{destino}/dia-07-csv")

PASTA_NB = str(Path.cwd())          # usada pelas células para voltar sempre à pasta certa
print("📂 Pasta do dia:", PASTA_NB)
print("🐍 Python", sys.version.split()[0])

## 🎯 Objetivos

- Entender o formato **CSV** e por que no Brasil ele costuma usar `;`.
- Rodar os exemplos originais: escrever, ler, ler uma coluna e converter JSON em CSV.
- Ler a planilha do caso com `csv.DictReader` e somar dinheiro com `Decimal`.
- Baixar os municípios de MG da **API do IBGE** e salvar em CSV que o Excel abre direito.
- Padronizar cidades digitadas à mão com `unicodedata`, apelidos e `difflib`.

## 🧠 Conceito em 2 minutos

**CSV** (*Comma-Separated Values*) é uma tabela em texto puro: cada linha é uma linha da planilha, e
um separador divide as colunas. Em inglês o separador é a vírgula. No Brasil a vírgula é o separador
**decimal** (R$ 149,90), então as planilhas usam **ponto e vírgula**.

```
id;data;cidade;valor
DEN-001;2025-11-30;Governador Valadares;149.90
```

Três detalhes que derrubam quem está começando:

| Problema | Solução |
|---|---|
| Linhas em branco extras no Windows | `open(..., newline="")` |
| Acentos viram `Ã§` no Excel | grave com `encoding="utf-8-sig"` |
| `0.1 + 0.2 = 0.30000000000000004` | dinheiro com `Decimal`, nunca `float` |

`csv.reader` devolve cada linha como **lista** (`row[0]`). `csv.DictReader` devolve como
**dicionário** (`row["cidade"]`): o código fica legível e não quebra se alguém mudar a ordem das colunas.

## 🥋 Nível 1 — Exercício original (cipher387)

| Arquivo | O que faz |
|---|---|
| [`original/write_csv.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-07-csv/original/write_csv.py) | Cria `test.csv` com cabeçalho e uma linha, separados por `;` |
| [`original/read_csv.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-07-csv/original/read_csv.py) | Lê `test.csv` e imprime cada linha |
| [`original/read_csv_one_column.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-07-csv/original/read_csv_one_column.py) | Imprime só a primeira coluna |
| [`original/json_to_csv.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-07-csv/original/json_to_csv.py) | Converte a busca do GitHub (JSON) em CSV: login, perfil e avatar |

```bash
pip install requests
cd dia-07-csv/original
python write_csv.py
python read_csv.py
python read_csv_one_column.py
python json_to_csv.py          # sobrescreve o test.csv com os usuários do GitHub
```

> 💡 Rode de **dentro** de `original/`: os scripts gravam e leem `test.csv` na pasta atual. O autor
> também lembra que, muitas vezes, um conversor online (como o csvjson.com) resolve mais rápido do
> que escrever código. Vale para uma vez; para repetir toda semana, o script ganha.

In [ ]:
#@title 📦 Instalar as bibliotecas do exercício original
%pip install -q requests

### `original/json_to_csv.py` — Converter a resposta JSON da API do GitHub em CSV

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Converter a resposta JSON da API do GitHub em CSV
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_7/json_to_csv.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: pip install requests
# ──────────────────────────────────────────────────────────────────────────

import json
import requests
import csv


# 💬 O mesmo pedido dos Dias 5 e 6.
response = requests.get("https://api.github.com/search/users?q=javascript")

json_data=response.json()


# 💬 Atenção: isto SOBRESCREVE o test.csv criado pelo write_csv.py.
csv_file = open('test.csv', 'w')

writer = csv.writer(csv_file, delimiter =';')

# 💬 O mesmo "- 1" do Dia 6: o último usuário da página fica de fora.
usersCount = len(json_data['items'])-1

# 💬 Para cada usuário, monta uma linha com 3 colunas: login, perfil e foto.
for x in range(usersCount):
    row = []
    row.append(json_data['items'][x]['login'])
    row.append(json_data['items'][x]['html_url'])
    row.append(json_data['items'][x]['avatar_url'])
    writer.writerow(row)
   

csv_file.close()

### `original/read_csv.py` — Ler um arquivo CSV linha por linha

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Ler um arquivo CSV linha por linha
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_7/read_csv.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: rode antes original/write_csv.py (ou json_to_csv.py) para criar test.csv.
# ──────────────────────────────────────────────────────────────────────────

import csv

# 💬 "with" abre o arquivo e fecha sozinho no final do bloco. 'r' = só leitura.
with open("test.csv", 'r') as csv_file:
  # 💬 Sem delimiter, o leitor espera VÍRGULA. Como o arquivo usa ";", cada linha
  # 💬 vem como uma lista com um único texto: ['Smith;John;35;USA'].
  csv_reader = csv.reader(csv_file)
  for row in csv_reader:
    print(row)

### `original/read_csv_one_column.py` — Ler só a primeira coluna de um CSV

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Ler só a primeira coluna de um CSV
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_7/read_csv_one_column.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: rode antes original/write_csv.py para criar test.csv.
# ──────────────────────────────────────────────────────────────────────────

import csv

with open("test.csv", 'r') as csv_file:
  csv_reader = csv.reader(csv_file)
  for row in csv_reader: 
    # 💬 Como o leitor usou vírgula, row[0] é a linha inteira; o autor separa no ";" na mão.
    # 💬 Funciona, mas quebra se algum campo tiver ";" dentro de aspas. A forma robusta é
    # 💬 csv.reader(csv_file, delimiter=';') e depois row[0] (veja br/01_ler_denuncias.py).
    columns=row[0].split(";")
    print(columns[0])

### `original/write_csv.py` — Criar um arquivo CSV com cabeçalho e uma linha de dados

In [ ]:
%cd -q {PASTA_NB}/original
# ──────────────────────────────────────────────────────────────────────────
# Título: Criar um arquivo CSV com cabeçalho e uma linha de dados
# Código ORIGINAL de cipher387 (Cyb_detective), "Python for OSINT. 21-day course"
# Fonte: https://github.com/cipher387/python-for-OSINT-21-days/blob/main/Day_7/write_csv.py
# Licença: MIT © 2023 Cyb_detective. O código está intacto; só foram
# acrescentados comentários em pt-BR (linhas que começam com "# 💬").
# Requer: nada (csv vem com o Python). Rode de dentro de original/: cria test.csv ali.
# ──────────────────────────────────────────────────────────────────────────

# 💬 O módulo csv vem com o Python.
import csv

# 💬 Abre (e cria) test.csv para escrita ("w" apaga o conteúdo anterior!).
# 💬 Dica 2026: no Windows, acrescente newline="" e encoding="utf-8" na sua versão,
# 💬 senão aparecem linhas em branco extras e acentos quebrados.
csv_file = open('test.csv', 'w')

# 💬 O "escritor" de CSV. delimiter=';' usa ponto e vírgula, como o Excel em português gosta.
writer = csv.writer(csv_file, delimiter =';')

# 💬 Uma lista = uma linha. Primeiro os nomes das colunas...
header = ['Last name', 'First name', 'Age', 'Country']

# 💬 ...depois os dados (exemplo do autor).
data = ['Smith', 'John', '35', 'USA']

writer.writerow(header)

writer.writerow(data)

# 💬 Fechar o arquivo garante que tudo foi gravado no disco.
csv_file.close()

## 🇧🇷 Nível 2 — Camada Brasil

Tudo com a **biblioteca padrão**. Amostra offline do IBGE em `dados/`.

| Arquivo | O que faz |
|---|---|
| [`br/01_ler_denuncias.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-07-csv/br/01_ler_denuncias.py) | Lê `denuncias.csv` com `DictReader` e `;`, soma com `Decimal` e agrupa por produto e chave PIX |
| [`br/02_ibge_municipios.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-07-csv/br/02_ibge_municipios.py) | Baixa os municípios de MG do IBGE e grava `saidas/municipios_mg.csv` (`utf-8-sig`) |
| [`br/03_padroniza_cidades.py`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/dia-07-csv/br/03_padroniza_cidades.py) | **O caso:** padroniza as cidades (acentos, maiúsculas, apelidos, erros de digitação) e grava `saidas/denuncias_padronizadas.csv` |

```bash
cd dia-07-csv
python br/01_ler_denuncias.py
python br/02_ibge_municipios.py
OSINT_OFFLINE=1 python br/03_padroniza_cidades.py     # força o modo offline
```

**O que você deve ver no caso:** prejuízo total de **R$ 54.877,40**. A coluna de cidade cai de
**16 grafias diferentes para 11 cidades reais**, e Belo Horizonte, antes espalhada em três grafias,
aparece com **8 vítimas**, a maior concentração.

> ⚠️ **Campos `null` na API do IBGE.** Alguns municípios vêm com `"microrregiao": null`. Atenção:
> `m.get("microrregiao", {})` **não** resolve, porque a chave existe e o valor é `None`. Use
> `(m.get("microrregiao") or {})` ou uma função que ande pelo caminho checando cada degrau (como
> a `caminho()` do script 02). A amostra offline simula isso em "Belo Oriente".

> 💡 Os campos com formulário à direita (`#@param`) podem ser editados sem mexer no código. Para testar sem internet, rode antes: `os.environ['OSINT_OFFLINE'] = '1'`.

### `br/01_ler_denuncias.py` — Ler a planilha de denúncias (CSV com ";") e somar o prejuízo sem errar centavos.

In [ ]:
%cd -q {PASTA_NB}
"""
Ler a planilha de denúncias (CSV com ";") e somar o prejuízo sem errar centavos.

Carta 28 (rank B) — Jutsu da Planilha Viva

O que você aprende:
  • csv.DictReader: cada linha vira um dicionário {"coluna": "valor"};
  • delimiter=";" (padrão das planilhas brasileiras) e encoding="utf-8";
  • por que somar dinheiro com Decimal e não com float;
  • a agrupar valores por produto com um dicionário.

Como rodar (dentro da pasta dia-07-csv):
    python br/01_ler_denuncias.py
"""

import csv
from collections import defaultdict   # dicionário que cria o valor inicial sozinho
from decimal import Decimal           # números decimais exatos (dinheiro!)
from pathlib import Path

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"


def reais(valor):
    """Decimal('54877.40') → 'R$ 54.877,40' (formato brasileiro, sem depender de locale)."""
    return "R$ " + f"{valor:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")


# ── 1. Abrir e ler ────────────────────────────────────────────────────────
# newline="" é a recomendação oficial do módulo csv (evita problemas com quebras de linha).
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8", newline="") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";")
    colunas = leitor.fieldnames          # a primeira linha vira os nomes das colunas
    denuncias = list(leitor)             # list() lê todas as linhas de uma vez

print(f"📄 {len(denuncias)} denúncias · colunas: {', '.join(colunas)}\n")
print(f"🔎 Primeira linha como dicionário:\n   {dict(denuncias[0])}\n")

# ── 2. A armadilha do float ───────────────────────────────────────────────
# O computador guarda float em binário e alguns centavos "escorregam".
soma_float = sum(float(d["valor"]) for d in denuncias)
soma_decimal = sum(Decimal(d["valor"]) for d in denuncias)   # "149.90" vira Decimal exato
print(f"🧮 Soma com float:   {soma_float!r}")
print(f"🧮 Soma com Decimal: {soma_decimal!r}   ← use este para dinheiro")
# Aqui o float até acertou (por sorte do arredondamento na hora de imprimir), mas veja:
print(f"   Prova do problema: 0.1 + 0.2 com float = {0.1 + 0.2!r}; com Decimal = {Decimal('0.1') + Decimal('0.2')!r}\n")

# ── 3. Total e quantidade por produto ─────────────────────────────────────
total_produto = defaultdict(Decimal)    # valor inicial: Decimal('0')
qtd_produto = defaultdict(int)          # valor inicial: 0
for d in denuncias:
    total_produto[d["produto"]] += Decimal(d["valor"])
    qtd_produto[d["produto"]] += 1

print("📦 Prejuízo por produto (do maior para o menor):")
# sorted com key: ordena pelos valores; reverse=True = do maior para o menor
for produto, total in sorted(total_produto.items(), key=lambda par: par[1], reverse=True):
    print(f"   {produto:<18} {qtd_produto[produto]:>2} vítima(s)  {reais(total):>14}")
print(f"   {'TOTAL':<18} {len(denuncias):>2} vítima(s)  {reais(soma_decimal):>14}")

# ── 4. Uma pergunta rápida: quantas denúncias por chave PIX? ──────────────
por_chave = defaultdict(int)
for d in denuncias:
    por_chave[d["chave_pix_destino"]] += 1
print("\n🔑 Denúncias por chave PIX de destino:")
for chave, qtd in sorted(por_chave.items(), key=lambda par: -par[1]):
    print(f"   {chave}  {qtd:>2}")

### `br/02_ibge_municipios.py` — Baixar a lista oficial de municípios de MG (API do IBGE) e salvar em CSV.

In [ ]:
%cd -q {PASTA_NB}
"""
Baixar a lista oficial de municípios de MG (API do IBGE) e salvar em CSV.

Carta 28 (rank B) — Jutsu da Planilha Viva (parte 2)

O que você aprende:
  • a transformar JSON aninhado em linhas de planilha (o "json_to_csv" do original, em versão BR);
  • a se proteger de campos que vêm null ("microrregiao": null em alguns municípios);
  • csv.DictWriter com delimiter=";" e encoding="utf-8-sig" (o Excel em português abre certinho).

Como rodar (dentro da pasta dia-07-csv):
    python br/02_ibge_municipios.py
    OSINT_OFFLINE=1 python br/02_ibge_municipios.py
"""

import csv
import gzip
import json
import os
import urllib.request
from pathlib import Path

UF = "MG"  #@param {type:"string"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"
SAIDAS = PASTA_DIA / "saidas"



def ler_resposta(resposta):
    """Lê a resposta HTTP e devolve o texto, descompactando se vier em gzip.

    O IBGE responde COMPACTADO (gzip) mesmo quando não pedimos. O urllib não
    descompacta sozinho, então olhamos os 2 primeiros bytes: 0x1f 0x8b é a
    "assinatura" de todo arquivo gzip (os números mágicos que você vê no Dia 14).
    """
    dados = resposta.read()
    if dados[:2] == b"\x1f\x8b" or resposta.headers.get("Content-Encoding") == "gzip":
        dados = gzip.decompress(dados)
    return dados.decode("utf-8")

def buscar_json(url, amostra):
    """Baixa um JSON da internet; se falhar (ou OFFLINE=1), lê a amostra local."""
    if not OFFLINE:
        try:
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=30) as resposta:
                return json.loads(ler_resposta(resposta))
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    return json.loads((PASTA_DIA / "dados" / amostra).read_text(encoding="utf-8"))


def caminho(dicionario, *chaves):
    """Anda por várias chaves sem quebrar: caminho(m, "microrregiao", "nome").

    Se qualquer degrau for None ou faltar, devolve "" em vez de dar erro.
    (dados.get("microrregiao", {}) NÃO basta: se a chave existe com valor null,
    o .get devolve None, e None.get(...) quebra.)
    """
    atual = dicionario
    for chave in chaves:
        if not isinstance(atual, dict):
            return ""
        atual = atual.get(chave)
    return atual if atual is not None else ""


# ── 1. Baixar ─────────────────────────────────────────────────────────────
url = f"https://servicodados.ibge.gov.br/api/v1/localidades/estados/{UF}/municipios"
resposta = buscar_json(url, "ibge_municipios_mg.json")
# A API real devolve uma LISTA; a amostra guarda a lista dentro de "municipios" (para ter o _aviso).
municipios = resposta if isinstance(resposta, list) else resposta.get("municipios", [])
print(f"🏙️  {len(municipios)} municípios recebidos para {UF}")

# ── 2. Achatar: cada município vira UMA linha com colunas simples ─────────
linhas = []
sem_micro = []
for m in municipios:
    if m.get("microrregiao") is None:
        sem_micro.append(m.get("nome"))
    linhas.append({
        "codigo_ibge": m.get("id"),
        "municipio": m.get("nome"),
        "microrregiao": caminho(m, "microrregiao", "nome"),
        "mesorregiao": caminho(m, "microrregiao", "mesorregiao", "nome"),
        # Divisão regional mais nova do IBGE (2017): regiões imediatas e intermediárias.
        "regiao_imediata": caminho(m, "regiao-imediata", "nome"),
        "regiao_intermediaria": caminho(m, "regiao-imediata", "regiao-intermediaria", "nome"),
    })

if sem_micro:
    print(f"⚠️  Vieram com microrregiao = null: {', '.join(sem_micro)} (nosso código não quebrou 😉)")

# ── 3. Gravar o CSV ───────────────────────────────────────────────────────
SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / f"municipios_{UF.lower()}.csv"
# utf-8-sig coloca uma "assinatura" (BOM) no início: é o sinal para o Excel ler como UTF-8.
with open(destino, "w", encoding="utf-8-sig", newline="") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=list(linhas[0]), delimiter=";")
    escritor.writeheader()
    escritor.writerows(linhas)
print(f"💾 {destino.relative_to(PASTA_DIA)} gravado com {len(linhas)} linhas.\n")

# ── 4. Conferir: reler as 5 primeiras linhas ──────────────────────────────
with open(destino, encoding="utf-8-sig", newline="") as arquivo:
    for i, linha in enumerate(csv.DictReader(arquivo, delimiter=";")):
        if i == 5:
            break
        print(f"   {linha['codigo_ibge']}  {linha['municipio']:<22} {linha['mesorregiao']}")

### `br/03_padroniza_cidades.py` — Padronizar as cidades digitadas pelas vítimas usando a lista oficial do IBGE.

In [ ]:
%cd -q {PASTA_NB}
"""
Padronizar as cidades digitadas pelas vítimas usando a lista oficial do IBGE.

Carta 28 (rank B) — Jutsu da Planilha Viva (aplicação no caso)

O que você aprende:
  • por que "contagem", "Belo Horizone" e "Pocos de Caldas" atrapalham qualquer contagem;
  • a normalizar texto: tirar acentos (unicodedata) + casefold + espaços;
  • a usar uma tabela de apelidos ("BH" → Belo Horizonte);
  • a corrigir erros de digitação com difflib.get_close_matches (aproximação de texto);
  • a gravar a planilha padronizada com o código IBGE e mostrar o "antes → depois".

Como rodar (dentro da pasta dia-07-csv):
    python br/03_padroniza_cidades.py
    OSINT_OFFLINE=1 python br/03_padroniza_cidades.py
(Se você rodou antes o 02_ibge_municipios.py, ele aproveita saidas/municipios_mg.csv.)
"""

import csv
import difflib              # compara textos parecidos ("Belo Horizone" ≈ "Belo Horizonte")
import gzip
import json
import os
import unicodedata          # separar letras de acentos
import urllib.request
from collections import Counter
from pathlib import Path

SIMILARIDADE_MINIMA = 0.85  #@param {type:"number"}

PASTA_DIA = Path(__file__).resolve().parent.parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"

# Apelidos e siglas que a gente ouve no dia a dia (chave já normalizada).
APELIDOS = {"bh": "Belo Horizonte", "beaga": "Belo Horizonte", "jf": "Juiz de Fora",
            "gv": "Governador Valadares", "udia": "Uberlândia"}



def ler_resposta(resposta):
    """Lê a resposta HTTP e devolve o texto, descompactando se vier em gzip.

    O IBGE responde COMPACTADO (gzip) mesmo quando não pedimos. O urllib não
    descompacta sozinho, então olhamos os 2 primeiros bytes: 0x1f 0x8b é a
    "assinatura" de todo arquivo gzip (os números mágicos que você vê no Dia 14).
    """
    dados = resposta.read()
    if dados[:2] == b"\x1f\x8b" or resposta.headers.get("Content-Encoding") == "gzip":
        dados = gzip.decompress(dados)
    return dados.decode("utf-8")

def normalizar(texto):
    """'  Poços  de Caldas ' → 'pocos de caldas'."""
    # NFKD separa "ç" em "c" + "cedilha"; depois jogamos fora o que não é ASCII (os acentos).
    sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    # casefold é um lower() mais forte; split/join junta espaços repetidos.
    return " ".join(sem_acento.casefold().split())


def carregar_municipios():
    """Devolve {codigo_ibge: nome_oficial}. Ordem de preferência: CSV do script 02, API, amostra."""
    csv_02 = SAIDAS / "municipios_mg.csv"
    if csv_02.exists():
        with open(csv_02, encoding="utf-8-sig", newline="") as arquivo:
            return {int(l["codigo_ibge"]): l["municipio"] for l in csv.DictReader(arquivo, delimiter=";")}
    lista = None
    if not OFFLINE:
        try:
            url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/MG/municipios"
            pedido = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(pedido, timeout=30) as resposta:
                lista = json.loads(ler_resposta(resposta))
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    if lista is None:
        lista = json.loads((PASTA_DIA / "dados" / "ibge_municipios_mg.json").read_text(encoding="utf-8"))["municipios"]
    return {m["id"]: m["nome"] for m in lista}


# ── 1. Base oficial, indexada pelo nome normalizado ───────────────────────
municipios = carregar_municipios()
por_nome = {normalizar(nome): (codigo, nome) for codigo, nome in municipios.items()}
print(f"📚 Base oficial: {len(municipios)} municípios de MG\n")


def padronizar(cidade_digitada):
    """Devolve (nome_oficial, codigo_ibge, metodo). metodo explica COMO chegamos lá."""
    chave = normalizar(cidade_digitada)
    if chave in por_nome:                                      # 1º: igual depois de normalizar
        codigo, nome = por_nome[chave]
        return nome, codigo, "exato" if nome == cidade_digitada else "normalizado"
    if chave in APELIDOS:                                      # 2º: apelido conhecido
        codigo, nome = por_nome[normalizar(APELIDOS[chave])]
        return nome, codigo, "apelido"
    parecidos = difflib.get_close_matches(chave, por_nome, n=1, cutoff=SIMILARIDADE_MINIMA)
    if parecidos:                                              # 3º: erro de digitação
        codigo, nome = por_nome[parecidos[0]]
        return nome, codigo, "aproximado"
    return "", "", "NÃO RECONHECIDA"                           # 4º: humano precisa olhar


# ── 2. Ler denúncias e padronizar ─────────────────────────────────────────
with open(PASTA_CASO / "denuncias.csv", encoding="utf-8", newline="") as arquivo:
    leitor = csv.DictReader(arquivo, delimiter=";")
    colunas = leitor.fieldnames
    denuncias = list(leitor)

mudancas = {}                                      # {"Belo Horizone": ("Belo Horizonte", "aproximado")}
for d in denuncias:
    nome, codigo, metodo = padronizar(d["cidade"])
    d["cidade_padronizada"], d["codigo_ibge"], d["metodo"] = nome, codigo, metodo
    if metodo != "exato":
        mudancas[d["cidade"]] = (nome, metodo)

# ── 3. Mostrar o "antes → depois" ─────────────────────────────────────────
print("✏️  Antes → depois:")
for antes, (depois, metodo) in sorted(mudancas.items()):
    print(f"   {antes!r:<24} → {depois or '???':<22} ({metodo})")

antes = Counter(d["cidade"] for d in denuncias)
depois = Counter(d["cidade_padronizada"] for d in denuncias)
print(f"\n📊 Cidades diferentes: {len(antes)} antes · {len(depois)} depois")
print("🏆 Ranking padronizado:")
for cidade, qtd in depois.most_common():
    print(f"   {cidade:<22} {'█' * qtd} {qtd}")

# ── 4. Gravar a planilha nova (com as colunas extras no final) ────────────
SAIDAS.mkdir(exist_ok=True)
destino = SAIDAS / "denuncias_padronizadas.csv"
with open(destino, "w", encoding="utf-8-sig", newline="") as arquivo:
    escritor = csv.DictWriter(arquivo, fieldnames=colunas + ["cidade_padronizada", "codigo_ibge", "metodo"],
                              delimiter=";")
    escritor.writeheader()
    escritor.writerows(denuncias)
print(f"\n💾 {destino.relative_to(PASTA_DIA)} gravado. A coluna original 'cidade' foi mantida:")
print("   nunca apague o dado bruto; acrescente a versão tratada ao lado.")

## ⚖️ Ética, LGPD e OPSEC

- **A planilha das vítimas é o arquivo mais sensível do caso.** Tem nome, CPF e telefone. Não envie
  por e-mail pessoal, não suba em planilha online pública e não cole em ferramentas de IA. Para
  análises que não precisam de identificação, trabalhe com uma cópia sem `nome_vitima`, `cpf` e
  `telefone_contato` (minimização, LGPD art. 6º, III).
- **Nunca sobrescreva o dado bruto.** Acrescente colunas tratadas ao lado (`cidade_padronizada`,
  `metodo`). Assim qualquer pessoa pode conferir o que você mudou, e por quê.
- **Correção automática erra.** `difflib` dá 71% de semelhança entre "uberaba" e "uberlandia": com um corte baixo (0,7), uma
  vítima de Uberaba que digitasse errado poderia "mudar de cidade".
  Os casos "aproximado" precisam de olho humano antes de irem para o relatório.
- **CSV abre fórmulas.** Um campo que começa com `=`, `+`, `-` ou `@` pode virar fórmula no Excel
  (*CSV injection*). Se a planilha vier de fonte não confiável, abra primeiro num editor de texto.

## 🎯 Quiz rápido

Escolha a resposta no formulário de cada célula e rode a célula.

In [ ]:
#@title 1. Por que o original read_csv.py imprime ['Smith;John;35;USA'] numa lista de um item só?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Porque o arquivo está corrompido", "B) Porque o csv.reader espera vírgula por padrão e o arquivo usa ponto e vírgula", "C) Porque faltou o encoding utf-8", "D) Porque o CSV só aceita uma coluna"]
correta = "B) Porque o csv.reader espera vírgula por padrão e o arquivo usa ponto e vírgula"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Sem delimiter=';', o leitor separa por vírgula; como não há vírgula, a linha inteira vira um campo só.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 2. Qual codificação faz o Excel em português abrir um CSV UTF-8 com acentos corretos?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) ascii", "B) latin-1", "C) utf-8-sig", "D) utf-16-be"]
correta = "C) utf-8-sig"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "utf-8-sig grava a assinatura BOM no início do arquivo, que avisa o Excel de que o texto é UTF-8.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

In [ ]:
#@title 3. O que difflib.get_close_matches('belo horizone', base, cutoff=0.85) faz?
resposta = "Escolha uma opção" #@param ["Escolha uma opção", "A) Apaga as cidades erradas", "B) Devolve o nome da base mais parecido, se a semelhança for de pelo menos 85%", "C) Traduz o nome da cidade", "D) Conta quantas letras são iguais"]
correta = "B) Devolve o nome da base mais parecido, se a semelhança for de pelo menos 85%"
if resposta == "Escolha uma opção":
    print("👆 Escolha uma opção no formulário e rode de novo.")
elif resposta == correta:
    print("✅ Acertou!", "Ele compara textos e devolve os mais parecidos acima do corte. Ótimo para erros de digitação, mas sempre revise os casos aproximados.")
else:
    print("❌ Ainda não. Releia o conceito do dia e tente outra vez.")

# 🏆 Desafio S — Dia 7: Resumo regional das denúncias

**Carta 49 · Rank S**

## A missão

A Delegada Rita ligou: o Ministério Público quer saber **em que regiões de Minas** o golpe pegou mais
gente, porque pode haver outras delegacias com o mesmo caso. Ela precisa de uma planilha que abra
direto no Excel, sem ninguém "arrumar" nada.

Escreva um script que:

1. Padronize a coluna `cidade` de `caso/dados/denuncias.csv` com a base do IBGE (como no script 03).
2. Descubra a **mesorregião** de cada cidade (campo `microrregiao → mesorregiao → nome`), sem quebrar
   quando `microrregiao` vier `null`.
3. Calcule, com `Decimal`, **vítimas e prejuízo por cidade** e **por mesorregião**.
4. Grave `saidas/resumo_por_cidade.csv` e `saidas/resumo_por_mesorregiao.csv` com `;`,
   `utf-8-sig` e **vírgula decimal** (`21441,30`), para o Excel brasileiro reconhecer como número.
5. Mostre um relatório de qualidade: quantas cidades estavam certas, quantas foram normalizadas,
   aproximadas ou não reconhecidas.
6. **Confira** que a soma por cidade bate com o total da planilha original.
7. Funcione **sem internet** com a amostra do IBGE.

## Critérios de rank S

- [ ] Só biblioteca padrão
- [ ] Nenhuma cidade some: as não reconhecidas aparecem marcadas para revisão humana
- [ ] A soma confere (✅) e os valores abrem como número no Excel em pt-BR
- [ ] A Região Metropolitana de BH aparece em primeiro lugar

## Dica

`(m.get("microrregiao") or {}).get("mesorregiao")`: o `or {}` troca `None` por um dicionário vazio,
e o `.get` seguinte não quebra.

In [ ]:
%cd -q {PASTA_NB}
# ✍️ Escreva sua solução aqui.

In [ ]:
#@title 🔒 Gabarito (dê dois cliques aqui para ver o código) { display-mode: "form" }
%cd -q {PASTA_NB}
"""
Gabarito do Desafio S do Dia 7 — Resumo regional das denúncias.

Junta a planilha de denúncias com a base de municípios do IBGE (cidade padronizada +
mesorregião), calcula vítimas e prejuízo por cidade e por mesorregião, e grava
dois CSVs prontos para o Excel em português, mais um relatório de qualidade
(quantas cidades foram corrigidas e como). Sem internet, usa a amostra do IBGE.
"""

import csv
import difflib
import gzip
import json
import os
import unicodedata
import urllib.request
from collections import Counter, defaultdict
from decimal import Decimal
from pathlib import Path

SIMILARIDADE_MINIMA = 0.85  #@param {type:"number"}

PASTA_DIA = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
PASTA_CASO = PASTA_DIA.parent / "caso" / "dados"
SAIDAS = PASTA_DIA / "saidas"
OFFLINE = os.environ.get("OSINT_OFFLINE") == "1"
APELIDOS = {"bh": "Belo Horizonte", "jf": "Juiz de Fora", "gv": "Governador Valadares"}



def ler_resposta(resposta):
    """Lê a resposta HTTP e devolve o texto, descompactando se vier em gzip.

    O IBGE responde COMPACTADO (gzip) mesmo quando não pedimos. O urllib não
    descompacta sozinho, então olhamos os 2 primeiros bytes: 0x1f 0x8b é a
    "assinatura" de todo arquivo gzip (os números mágicos que você vê no Dia 14).
    """
    dados = resposta.read()
    if dados[:2] == b"\x1f\x8b" or resposta.headers.get("Content-Encoding") == "gzip":
        dados = gzip.decompress(dados)
    return dados.decode("utf-8")

def normalizar(texto):
    sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("ascii")
    return " ".join(sem_acento.casefold().split())


def reais_excel(valor):
    """Decimal → '54877,40': vírgula decimal, sem separador de milhar, para o Excel BR somar."""
    return f"{valor:.2f}".replace(".", ",")


def municipios_ibge():
    lista = None
    if not OFFLINE:
        try:
            url = "https://servicodados.ibge.gov.br/api/v1/localidades/estados/MG/municipios"
            req = urllib.request.Request(url, headers={"User-Agent": "python-para-osint-21-dias/estudo"})
            with urllib.request.urlopen(req, timeout=30) as resposta:
                lista = json.loads(ler_resposta(resposta))
        except Exception as erro:
            print(f"⚠️ MODO OFFLINE: a consulta falhou ({erro}). Usando a amostra local.")
    else:
        print("⚠️ MODO OFFLINE: usando a amostra local.")
    if lista is None:
        lista = json.loads((PASTA_DIA / "dados" / "ibge_municipios_mg.json").read_text(encoding="utf-8"))["municipios"]
    base = {}
    for m in lista:
        micro = m.get("microrregiao") or {}                 # null → {}
        meso = (micro.get("mesorregiao") or {}).get("nome", "")
        base[normalizar(m["nome"])] = {"nome": m["nome"], "codigo": m["id"], "mesorregiao": meso or "(sem mesorregião)"}
    return base


base = municipios_ibge()

with open(PASTA_CASO / "denuncias.csv", encoding="utf-8", newline="") as f:
    denuncias = list(csv.DictReader(f, delimiter=";"))

metodos = Counter()
por_cidade = defaultdict(lambda: {"vitimas": 0, "prejuizo": Decimal("0"), "mesorregiao": "", "codigo": ""})
for d in denuncias:
    chave = normalizar(d["cidade"])
    if chave in base:
        metodo = "exato" if base[chave]["nome"] == d["cidade"] else "normalizado"
    elif chave in APELIDOS:
        chave, metodo = normalizar(APELIDOS[chave]), "apelido"
    else:
        parecido = difflib.get_close_matches(chave, base, n=1, cutoff=SIMILARIDADE_MINIMA)
        chave, metodo = (parecido[0], "aproximado") if parecido else (None, "não reconhecida")
    metodos[metodo] += 1
    info = base.get(chave, {"nome": f"?? {d['cidade']}", "codigo": "", "mesorregiao": "(revisar)"})
    linha = por_cidade[info["nome"]]
    linha["vitimas"] += 1
    linha["prejuizo"] += Decimal(d["valor"])
    linha["mesorregiao"], linha["codigo"] = info["mesorregiao"], info["codigo"]

por_meso = defaultdict(lambda: {"vitimas": 0, "prejuizo": Decimal("0"), "cidades": set()})
for cidade, v in por_cidade.items():
    m = por_meso[v["mesorregiao"]]
    m["vitimas"] += v["vitimas"]
    m["prejuizo"] += v["prejuizo"]
    m["cidades"].add(cidade)

SAIDAS.mkdir(exist_ok=True)
with open(SAIDAS / "resumo_por_cidade.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["cidade", "codigo_ibge", "mesorregiao", "vitimas", "prejuizo_reais"])
    for cidade, v in sorted(por_cidade.items(), key=lambda p: -p[1]["prejuizo"]):
        w.writerow([cidade, v["codigo"], v["mesorregiao"], v["vitimas"], reais_excel(v["prejuizo"])])

with open(SAIDAS / "resumo_por_mesorregiao.csv", "w", encoding="utf-8-sig", newline="") as f:
    w = csv.writer(f, delimiter=";")
    w.writerow(["mesorregiao", "cidades", "vitimas", "prejuizo_reais"])
    for meso, v in sorted(por_meso.items(), key=lambda p: -p[1]["prejuizo"]):
        w.writerow([meso, ", ".join(sorted(v["cidades"])), v["vitimas"], reais_excel(v["prejuizo"])])

print("🗺️  Por mesorregião:")
for meso, v in sorted(por_meso.items(), key=lambda p: -p[1]["prejuizo"]):
    print(f"   {meso:<34} {v['vitimas']:>2} vítimas  R$ {reais_excel(v['prejuizo']):>10}")
print("\n🧹 Qualidade da coluna 'cidade':", dict(metodos))
total = sum(Decimal(d["valor"]) for d in denuncias)
conferido = sum(v["prejuizo"] for v in por_cidade.values())
print("✅ Total confere com a planilha." if total == conferido else "🚩 Total NÃO confere!")
print("💾 saidas/resumo_por_cidade.csv e saidas/resumo_por_mesorregiao.csv gravados.")

## 🔎 Inovação OSINT do dia

**API de Localidades do IBGE.** É oficial, gratuita e sem cadastro: todos os estados, municípios,
distritos e as duas divisões regionais (micro/mesorregiões e as regiões imediatas/intermediárias de
2017), sempre com o **código IBGE de 7 dígitos**. Esse código é a "chave universal" para cruzar dados
públicos: o ViaCEP devolve ele (Dia 5), o Querido Diário usa ele para filtrar diários oficiais (Dia 9).
Biblioteca Python simples: `urllib.request` + `json` + `csv` (nativos).

## 📚 Para ir além

- API de Localidades do IBGE: <https://servicodados.ibge.gov.br/api/docs/localidades>
- Módulo `csv` do Python (em português): <https://docs.python.org/pt-br/3/library/csv.html>
- Módulo `difflib`: <https://docs.python.org/pt-br/3/library/difflib.html>
- OWASP sobre CSV injection: <https://owasp.org/www-community/attacks/CSV_Injection>

## ✅ Checklist

- [ ] Rodei os quatro scripts originais e entendi por que o `read_csv.py` não separa as colunas
- [ ] Somei o prejuízo com `Decimal`
- [ ] Gerei o CSV de municípios e abri no Excel/LibreOffice sem acento quebrado
- [ ] Padronizei as cidades e conferi os casos "aproximado"
- [ ] Fiz o desafio S
- [ ] Anotei no meu [`DIARIO.md`](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/blob/main/DIARIO.md)

---

🎉 Terminou? [Registre sua entrega do Dia 7](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias/issues/new?template=entreguei-o-dia.yml) · Baseado em [cipher387/python-for-OSINT-21-days](https://github.com/cipher387/python-for-OSINT-21-days) (MIT) · Adaptação pt-BR: [Ridd1kulusC0d3r/python-para-osint-21-dias](https://github.com/Ridd1kulusC0d3r/python-para-osint-21-dias)